In [ ]:
# CELL 1 — Install Dependencies (LangChain + OpenAI + SQLite)
# ------------------------------------------------------------
# - Installs only the libraries needed for your simple, correct
#   architecture using initialize_agent + tools + memory.
# - Safe for API limits (no API request here).

!pip install -q --upgrade pip
!pip install -q langchain langchain-core langchain-community
!pip install -q openai
!pip install -q pydantic
!pip install -q python-dotenv
!pip install -q sqlite-utils

In [ ]:

# CELL 2 — Load OpenAI API Key & Initialize the LLM
# - Use Colab Secrets (Extensions -> Secrets) and set OPENAI_API_KEY there.
# - Uses a lightweight OpenAI chat model (gpt-4o-mini) to reduce token usage.
# - temperature=0 for deterministic outputs (helps routing + parsing).

import os
from google.colab import userdata
from langchain.chat_models import ChatOpenAI

# Load API key from Colab Secrets (create "openai_api_key" there)
os.environ["OPENAI_API_KEY"] = userdata.get("openai_api_key2")
if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Add OPENAI_API_KEY to Colab Secrets (Extensions → Secrets)")

# Initialize LangChain OpenAI chat model (minimal cost configuration)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=os.environ["OPENAI_API_KEY"])

print("CELL 2: OpenAI Chat model initialized (gpt-4o-mini, temperature=0).")

CELL 2: OpenAI Chat model initialized (gpt-4o-mini, temperature=0).


In [ ]:
# CELL 3 — Core LangChain Imports
# - Imports message classes (HumanMessage, AIMessage).
# - Imports @tool decorator for defining tools.
# - Imports Runnable system (RunnableLambda, RunnableBranch).
# - Imports ConversationBufferMemory (memory buffer).
# - Pydantic for structured tool inputs (optional).


from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

# Runnable system (your mentor said: "use runnable architecture")
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough

# Memory buffer for multi-turn chat
from langchain.memory import ConversationBufferMemory

# Prompts (used later by initialize_agent)
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

# Structured input model (optional, used for tool inputs)
from pydantic import BaseModel, Field

# Standard Python libraries
import sqlite3   # for SQLite long-term memory (Cell 6)
import json
import os

print("CELL 3: Core LangChain imports loaded successfully.")


CELL 3: Core LangChain imports loaded successfully.


In [ ]:
# CELL 4 — Define 4 Tools (Positive, Negative, Marks, Suicide)
# - Tools must be simple + deterministic.
# - Tools will be passed to initialize_agent later.
# - Follows LangChain @tool structure exactly.

from langchain_core.tools import tool
# 1) Positive Prompt Tool

@tool
def positive_prompt_tool(prompt: str) -> str:
    """Returns an encouraging rewrite of the text."""
    return f"Positive version: Keep going! {prompt} — you're doing great!"
# 2) Negative Prompt Tool

@tool
def negative_prompt_tool(prompt: str) -> str:
    """Returns a negative/exclusion-style rewrite."""
    return f"Negative version: Honestly, {prompt} sounds difficult and unlikely."


# 3) Student Marks Tool
# simple static DB for demo
STUDENT_MARKS_DB = {
    "Priya": {"English": 92, "Maths": 88, "Science": 95},
    "Amit": {"English": 78, "Maths": 81, "Science": 74},
    "Rahul": {"English": 67, "Maths": 72, "Science": 70},
}

@tool
def student_marks_tool(info: str) -> str:
    """
    Format: 'Name,Subject'
    Returns the real marks stored in our DB.
    """
    try:
        name, subject = [i.strip() for i in info.split(",")]
    except:
        return "Use format: Name,Subject"

    if name not in STUDENT_MARKS_DB:
        return f"No data found for {name}."

    if subject not in STUDENT_MARKS_DB[name]:
        return f"{name} has no marks stored for {subject}."

    marks = STUDENT_MARKS_DB[name][subject]

    # simple grading logic
    if marks >= 90:
        grade = "A+"
    elif marks >= 80:
        grade = "A"
    elif marks >= 70:
        grade = "B"
    else:
        grade = "C"

    return f"{name} scored {marks} in {subject} (Grade: {grade})."


# 4) Suicide Safety Tool
@tool
def suicide_related_tool(text: str) -> str:
    """Provides a safe response for suicide-related messages."""
    return (
        "I'm really sorry you're feeling this way. "
        "Please reach out to someone you trust or a local helpline immediately. "
        "You deserve support and you're not alone."
    )

# Group tools for the agent
tools = [
    positive_prompt_tool,
    negative_prompt_tool,
    student_marks_tool,
    suicide_related_tool
]

print("CELL 4: Tools defined successfully.")


CELL 4: Tools defined successfully.


In [ ]:

# CELL 5 — Router Logic (RunnableBranch + Keyword Detection)
# - Router decides which tool to trigger based on keywords.
# - Zero API usage (local, safe, fast).
# - RunnableLambda + RunnableBranch follows LangChain structure.
# - Output will tell initialize_agent which tool request to run.

# --- Keyword-based routing function ---
def get_route(text: str) -> str:
    t = text.lower().strip()

    # 1. Suicide safety tool (highest priority)
    crisis_words = ["suicide", "kill myself", "want to die", "end my life", "worthless"]
    if any(word in t for word in crisis_words):
        return "suicide_related_tool"

    # 2. Negative prompt tool
    if "negative prompt" in t or "avoid" in t or "exclude" in t:
        return "negative_prompt_tool"

    # 3. Student marks tool
    if any(word in t for word in ["mark", "marks", "score", "grade", "subject"]):
        return "student_marks_tool"

    # 4. Positive prompt tool
    positive_triggers = ["sad", "tired", "stressed", "feeling down"]
    if any(word in t for word in positive_triggers):
        return "positive_prompt_tool"

    # 5. Default → no tool (normal chat)
    return "no_tool"


# --- Wrap the router inside a RunnableLambda ---
router_runnable = RunnableLambda(lambda x: {"route": get_route(x["text"]), "text": x["text"]})

print("CELL 5: Router logic (Runnable-based) loaded successfully.")


CELL 5: Router logic (Runnable-based) loaded successfully.


In [ ]:
# CELL 6 — Memory System (SQLite Long-Term Memory + Buffer Memory)
# - ConversationBufferMemory: short-term memory (in this session).
# - SQLite DB: long-term memory stored across notebook restarts.
# - Saves facts (like “my name is…”) and past conversation pairs.

import sqlite3

# 1) SHORT-TERM MEMORY (Buffer)

session_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# 2) LONG-TERM MEMORY (SQLite)

DB_FILE = "memory.db"

# Create table if it doesn't exist
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS long_term_memory (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    key TEXT,
    value TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS conversation_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user TEXT,
    assistant TEXT
)
""")

conn.commit()


# Helper functions to store + load memory

def save_fact(key: str, value: str):
    """Stores or updates a fact about the user in SQLite."""
    cursor.execute("DELETE FROM long_term_memory WHERE key=?", (key,))
    cursor.execute("INSERT INTO long_term_memory (key, value) VALUES (?, ?)", (key, value))
    conn.commit()

def get_all_facts() -> dict:
    """Returns all stored facts as a dictionary."""
    cursor.execute("SELECT key, value FROM long_term_memory")
    rows = cursor.fetchall()
    return {key: value for key, value in rows}

def save_conversation(user_msg: str, ai_msg: str):
    """Stores conversation turns in SQLite."""
    cursor.execute(
        "INSERT INTO conversation_history (user, assistant) VALUES (?, ?)",
        (user_msg, ai_msg)
    )
    conn.commit()


print("CELL 6: SQLite long-term memory + session buffer memory initialized successfully.")


CELL 6: SQLite long-term memory + session buffer memory initialized successfully.


In [ ]:
# CELL 7 — Create Main Agent Using initialize_agent
# - This is the ONLY agent we need (simple architecture).
# - initialize_agent handles:
#       ✓ tool calling
#       ✓ function calling
#       ✓ planning
#       ✓ chain-of-thought (hidden)
#       ✓ memory buffer usage
# - Router will decide WHEN to call this agent.

from langchain.agents import initialize_agent, AgentType

# Create the main agent with memory + tools
main_agent = initialize_agent(
    tools=tools,                 # tools from Cell 4
    llm=llm,                     # ChatOpenAI from Cell 2
    agent=AgentType.OPENAI_FUNCTIONS,
    memory=session_memory,       # short-term memory buffer
    verbose=True                 # prints reasoning steps (optional)
)

print("CELL 7: Main agent created successfully using initialize_agent.")


CELL 7: Main agent created successfully using initialize_agent.


In [ ]:
# CELL 8 — Main Pipeline (Routing + Agent + Memory)
# - Router decides which TOOL or NORMAL CHAT is needed.
# - main_agent handles tool-calling via OPENAI_FUNCTIONS.
# - SQLite stores facts (e.g., "my name is ...") and messages.
# - BufferMemory keeps short-term conversation context.

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

def run_pipeline(user_input: str) -> str:


    # 1. Save user facts to SQLite (example: "my name is ...")
    # --------------------------------------------------------
    text = user_input.lower()
    if "my name is" in text:
        try:
            name = text.split("my name is")[-1].strip().split()[0]
            save_fact("name", name)
        except:
            pass

    # Load all saved facts from SQLite
    facts = get_all_facts()      # returns dict: { key: value }

    # --------------------------------------------------------
    # 2. ROUTER decides what to do
    # --------------------------------------------------------
    route = get_route(user_input)
    print(f"[Router Selected]: {route}")

    # --------------------------------------------------------
    # 3. Build system message including saved facts
    # --------------------------------------------------------
    system_context = f"Known user facts: {facts}"

    # --------------------------------------------------------
    # 4. Ask agent to respond (it will call a tool if needed)
    # --------------------------------------------------------
    try:
        response = main_agent.run(
            f"{system_context}\nUser: {user_input}"
        )
    except Exception as e:
        print("⚠️ Agent error:", e)
        response = "Sorry, something went wrong while generating your response."

    # --------------------------------------------------------
    # 5. Save the conversation to SQLite (long-term memory)
    # --------------------------------------------------------
    save_conversation(user_input, response)

    # --------------------------------------------------------
    # 6. Return assistant's final output
    # --------------------------------------------------------
    return response

print("CELL 8: Main pipeline created successfully.")


CELL 8: Main pipeline created successfully.


In [ ]:
# CELL 9 — Interactive Chat Loop (Final)
# - Lets you talk to the agent in real time.
# - Uses run_pipeline() which:
#       ✓ routes user input
#       ✓ calls initialize_agent
#       ✓ triggers tools when needed
#       ✓ stores memory in SQLite
#       ✓ stores session history in buffer memory

print("Chat started. Type 'exit' or 'quit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit"]:
        print("\nChat ended.")
        break

    try:
        output = run_pipeline(user_input)
        print("Assistant:", output, "\n")
    except Exception as e:
        print("ERROR:", type(e).__name__, "-", str(e))
        continue

Chat started. Type 'exit' or 'quit' to stop.

You: hai
[Router Selected]: no_tool


> Entering new AgentExecutor chain...
Hello Karthik! How can I assist you today?

> Finished chain.
Assistant: Hello Karthik! How can I assist you today? 

You: my name is karthik
[Router Selected]: no_tool


> Entering new AgentExecutor chain...
Hello Karthik! How can I assist you today?

> Finished chain.
Assistant: Hello Karthik! How can I assist you today? 

You: i am feling sad
[Router Selected]: positive_prompt_tool


> Entering new AgentExecutor chain...

Invoking: `suicide_related_tool` with `{'text': 'i am feling sad'}`


I'm really sorry you're feeling this way. Please reach out to someone you trust or a local helpline immediately. You deserve support and you're not alone.I'm really sorry to hear that you're feeling this way, Karthik. It's important to talk to someone who can help. Please reach out to a friend, family member, or a local helpline. You deserve support, and you're not alone in th